In [12]:
import pandas as pd
import os
from pathlib import Path

# Define the path to the folder containing Excel files
# This looks for a 'private/sent' folder relative to where this script is located
excel_folder = Path('./private/sent/')

# Initialize an empty list to store all dataframes
# We'll combine them all at the end
all_dataframes = []

# Clear the log file if it exists from a previous run
log_file_path = 'missing_columns_log.txt'
if os.path.exists(log_file_path):
    os.remove(log_file_path)
    print(f"Cleared previous log file: {log_file_path}\n")

# Check if the folder exists
if not excel_folder.exists():
    print(f"Error: Folder '{excel_folder}' does not exist!")
    exit()

# Get all Excel files in the folder
# This finds all files ending with .xlsx or .xls
excel_files = list(excel_folder.glob('*.xlsx')) + list(excel_folder.glob('*.xls'))

if not excel_files:
    print(f"No Excel files found in '{excel_folder}'")
    exit()

print(f"Found {len(excel_files)} Excel file(s) to process\n")

# Process each Excel workbook
for file_path in excel_files:
    print(f"Processing: {file_path.name}")
    
    try:
        # Load the Excel file
        # excel_file is a dictionary where keys are sheet names
        excel_file = pd.ExcelFile(file_path)
        
        # Process each sheet in the current workbook
        for sheet_name in excel_file.sheet_names:
            print(f"  - Reading sheet: {sheet_name}")
            
            try:
                # Read the current sheet into a dataframe
                # nrows=14 means we only read the first 14 data rows (plus the header row)
                df = pd.read_excel(file_path, sheet_name=sheet_name, nrows=14)
                
                # Get list of all columns in this sheet
                available_columns = df.columns.tolist()
                
                # Define the columns we want to extract
                # Get columns at index 1, 2, 3 (which are the 2nd, 3rd, and 4th columns)
                # We use slicing [1:4] to get columns at positions 1, 2, and 3
                columns_2_3_4 = available_columns[1:4] if len(available_columns) >= 4 else available_columns[1:]
                
                # Define the specific named columns we always want
                required_columns = ["Bemerkungen", "# Eier", "# Eier Mitglieder"]
                
                # Build the final list of columns to extract
                # Start with columns 2, 3, and 4
                columns_to_extract = columns_2_3_4.copy()
                
                # Track which required columns are missing
                missing_columns = []
                
                # Add the required columns if they exist in this sheet
                for col in required_columns:
                    if col in available_columns and col not in columns_to_extract:
                        columns_to_extract.append(col)
                    elif col not in available_columns:
                        # This required column is missing from this sheet
                        missing_columns.append(col)
                
                # Log if any required columns are missing
                if missing_columns:
                    warning_msg = f"    ⚠ Missing columns: {', '.join(missing_columns)}"
                    print(warning_msg)
                    # Also write to a log file
                    with open('missing_columns_log.txt', 'a', encoding='utf-8') as log_file:
                        log_file.write(f"File: {file_path.name} | Sheet: {sheet_name} | Missing: {', '.join(missing_columns)}\n")
                
                # Extract only the columns we want
                df_filtered = df[columns_to_extract].copy()
                
                # Add metadata columns to track where this data came from
                # This helps you identify the source later
                df_filtered['source_file'] = file_path.name
                df_filtered['source_sheet'] = sheet_name
                
                # Add this filtered dataframe to our list
                all_dataframes.append(df_filtered)
                
                print(f"    ✓ Extracted {len(df_filtered)} rows and {len(columns_to_extract)} columns")
                
            except Exception as e:
                # If there's an error reading a specific sheet, print it but continue
                print(f"    ✗ Error reading sheet '{sheet_name}': {str(e)}")
                continue
    
    except Exception as e:
        # If there's an error reading the entire file, print it but continue
        print(f"  ✗ Error reading file '{file_path.name}': {str(e)}")
        continue

# Check if we successfully loaded any data
if not all_dataframes:
    print("\nNo data was successfully loaded!")
    exit()

# Combine all dataframes into a single dataframe
# ignore_index=True creates a new sequential index
print(f"\nCombining {len(all_dataframes)} sheet(s) into a single DataFrame...")
combined_df = pd.concat(all_dataframes, ignore_index=True)

print(f"\n{'='*60}")
print(f"SUCCESS! Combined DataFrame created")
print(f"{'='*60}")
print(f"Total rows: {len(combined_df)}")
print(f"Total columns: {len(combined_df.columns)}")
print(f"\nColumn names:\n{combined_df.columns.tolist()}")

# Check if any missing columns were logged
if os.path.exists(log_file_path):
    print(f"\n⚠ Some sheets were missing required columns.")
    print(f"Check '{log_file_path}' for details.")
else:
    print(f"\n✓ All sheets contained all required columns!")

# Display first few rows as a preview
print(f"\nFirst 5 rows preview:")
print(combined_df.head())

# Optional: Save the combined dataframe to a new Excel file
# Uncomment the line below if you want to save the result
# combined_df.to_excel('combined_output.xlsx', index=False)

# Optional: Save as CSV (often easier to work with)
# Uncomment the line below if you want to save as CSV
# combined_df.to_csv('combined_output.csv', index=False)

# The combined_df variable now contains all your data
# You can continue working with it in your analysis

Found 12 Excel file(s) to process

Processing: Einsatzplan Huehnergruppe KW 19 bis 39.xlsx
  - Reading sheet: Woche 19
    ✓ Extracted 14 rows and 6 columns
  - Reading sheet: Woche 20
    ✓ Extracted 14 rows and 6 columns
  - Reading sheet: Woche 21
    ✓ Extracted 14 rows and 6 columns
  - Reading sheet: Woche 22
    ✓ Extracted 14 rows and 6 columns
  - Reading sheet: Woche 23
    ✓ Extracted 14 rows and 6 columns
  - Reading sheet: Woche 24
    ✓ Extracted 14 rows and 6 columns
  - Reading sheet: Woche 25
    ✓ Extracted 14 rows and 6 columns
  - Reading sheet: Woche 26
    ✓ Extracted 14 rows and 6 columns
  - Reading sheet: Woche 27
    ✓ Extracted 14 rows and 6 columns
  - Reading sheet: Woche 28
    ✓ Extracted 14 rows and 6 columns
  - Reading sheet: Woche 29
    ✓ Extracted 14 rows and 6 columns
  - Reading sheet: Woche 30
    ✓ Extracted 14 rows and 6 columns
  - Reading sheet: Woche 31
    ✓ Extracted 14 rows and 6 columns
  - Reading sheet: Woche 32
    ✓ Extracted 14 rows

In [7]:
combined_df.to_csv('combined_output.csv', index=False)

In [13]:
print(f"\n{'='*60}")
print(f"DATA CLEANING AND TRANSFORMATION")
print(f"{'='*60}")

# Get the current column names to work with
current_columns = combined_df.columns.tolist()

# Rename the first three columns to Date, Day, AM/PM
# We need to identify which columns are at positions 0, 1, 2
if len(current_columns) >= 3:
    # Create a rename dictionary mapping old names to new names
    rename_dict = {
        current_columns[0]: 'Date',
        current_columns[1]: 'Day',
        current_columns[2]: 'AM/PM'
    }
    combined_df.rename(columns=rename_dict, inplace=True)
    print("✓ Renamed first three columns to: Date, Day, AM/PM")

# Remove the # sign from column names "# Eier" and "# Eier Mitglieder"
# This creates a new dictionary for all columns that need renaming
rename_dict_eier = {}
if '# Eier' in combined_df.columns:
    rename_dict_eier['# Eier'] = 'Eier'
if '# Eier Mitglieder' in combined_df.columns:
    rename_dict_eier['# Eier Mitglieder'] = 'Eier Mitglieder'

if rename_dict_eier:
    combined_df.rename(columns=rename_dict_eier, inplace=True)
    print(f"✓ Removed # from column names: {', '.join(rename_dict_eier.values())}")

# Convert Date column to datetime
# errors='coerce' means invalid dates will become NaT (Not a Time)
if 'Date' in combined_df.columns:
    combined_df['Date'] = pd.to_datetime(combined_df['Date'], errors='coerce')
    print("✓ Converted 'Date' column to datetime")

# Fill empty Date fields with the date from the row above
# ffill() stands for "forward fill" - it fills NaN/NaT values with the last valid value
if 'Date' in combined_df.columns:
    combined_df['Date'] = combined_df['Date'].ffill()
    print("✓ Filled empty Date fields with previous row's date")

# Fill empty Day fields with the day from the row above
if 'Day' in combined_df.columns:
    combined_df['Day'] = combined_df['Day'].ffill()
    print("✓ Filled empty Day fields with previous row's day")

# Convert Eier columns to integer
# First, we need to handle NaN values because integers can't be NaN in pandas
# We use Int64 (capital I) which is a nullable integer type
if 'Eier' in combined_df.columns:
    # Convert to numeric first (in case there are any string values)
    combined_df['Eier'] = pd.to_numeric(combined_df['Eier'], errors='coerce')
    # Convert to nullable integer type (Int64 allows NaN values)
    combined_df['Eier'] = combined_df['Eier'].astype('Int64')
    print("✓ Converted 'Eier' column to integer")

if 'Eier Mitglieder' in combined_df.columns:
    # Convert to numeric first (in case there are any string values)
    combined_df['Eier Mitglieder'] = pd.to_numeric(combined_df['Eier Mitglieder'], errors='coerce')
    # Convert to nullable integer type (Int64 allows NaN values)
    combined_df['Eier Mitglieder'] = combined_df['Eier Mitglieder'].astype('Int64')
    print("✓ Converted 'Eier Mitglieder' column to integer")

print(f"\n{'='*60}")
print(f"CLEANED DATA SUMMARY")
print(f"{'='*60}")
print(f"\nColumn names and types:")
print(combined_df.dtypes)
print(f"\nFirst 10 rows of cleaned data:")
print(combined_df.head(10))


DATA CLEANING AND TRANSFORMATION
✓ Renamed first three columns to: Date, Day, AM/PM
✓ Removed # from column names: Eier, Eier Mitglieder
✓ Converted 'Date' column to datetime
✓ Filled empty Date fields with previous row's date
✓ Filled empty Day fields with previous row's day
✓ Converted 'Eier' column to integer
✓ Converted 'Eier Mitglieder' column to integer

CLEANED DATA SUMMARY

Column names and types:
Date               datetime64[ns]
Day                        object
AM/PM                      object
Bemerkungen                object
Eier                        Int64
Eier Mitglieder             Int64
source_file                object
source_sheet               object
dtype: object

First 10 rows of cleaned data:
        Date Day AM/PM                                        Bemerkungen  \
0 2024-05-06  MO    am  Hühner und Mensch waren schon besser drauf. Es...   
1 2024-05-06  MO    pm                      Mo abend alles okay.  kein Ei   
2 2024-05-07  DI    am               Di m

In [9]:
combined_df.to_csv('combined_output.csv', index=False)

In [14]:
# Sort by Date and then by AM/PM
# This ensures all data is in chronological order
combined_df = combined_df.sort_values(by=['Date', 'AM/PM'], ascending=True)
# Reset the index after sorting so we have clean sequential indices
combined_df = combined_df.reset_index(drop=True)
print("✓ Sorted data by Date and AM/PM")

# Initialize a log file for duplicate conflicts
duplicates_log_path = 'duplicate_conflicts_log.txt'
if os.path.exists(duplicates_log_path):
    os.remove(duplicates_log_path)

# Find duplicates based on Date and AM/PM
# We'll process them group by group
duplicate_groups = combined_df.groupby(['Date', 'AM/PM'])

# Store indices of rows to delete
rows_to_delete = []

# Track if we found any conflicts
conflicts_found = False

# Process each group of potential duplicates
for (date, am_pm), group in duplicate_groups:
    # Only process if there are actually duplicates (more than 1 row)
    if len(group) > 1:
        print(f"\nProcessing duplicates for Date: {date}, AM/PM: {am_pm} ({len(group)} rows)")
        
        # Get the indices and Bemerkungen values for this group
        indices = group.index.tolist()
        bemerkungen_values = group['Bemerkungen'].tolist()
        
        # Check if Bemerkungen column exists
        if 'Bemerkungen' not in combined_df.columns:
            # If no Bemerkungen column, just keep the last duplicate
            rows_to_delete.extend(indices[:-1])
            print(f"  → No 'Bemerkungen' column found, keeping last row only")
            continue
        
        # Separate rows with empty vs filled Bemerkungen
        # pd.isna() checks for NaN, None, and empty strings
        empty_indices = [idx for idx, val in zip(indices, bemerkungen_values) 
                        if pd.isna(val) or (isinstance(val, str) and val.strip() == '')]
        filled_indices = [idx for idx, val in zip(indices, bemerkungen_values) 
                         if not (pd.isna(val) or (isinstance(val, str) and val.strip() == ''))]
        
        # Case 1: Some rows have empty Bemerkungen, some have filled
        if empty_indices and filled_indices:
            # Delete all rows with empty Bemerkungen
            rows_to_delete.extend(empty_indices)
            print(f"  → Case 1: Deleted {len(empty_indices)} row(s) with empty Bemerkungen")
            
            # Now check if remaining filled rows have conflicts
            if len(filled_indices) > 1:
                filled_values = [combined_df.loc[idx, 'Bemerkungen'] for idx in filled_indices]
                # Check if all filled values are identical
                if len(set(filled_values)) == 1:
                    # All identical, keep last one
                    rows_to_delete.extend(filled_indices[:-1])
                    print(f"  → Remaining filled rows are identical, keeping last one")
                else:
                    # Conflict: different Bemerkungen values - DO NOT DELETE
                    conflicts_found = True
                    with open(duplicates_log_path, 'a', encoding='utf-8') as log_file:
                        log_file.write(f"\nConflict at Date: {date}, AM/PM: {am_pm}\n")
                        for idx in filled_indices:
                            row_data = combined_df.loc[idx]
                            log_file.write(f"  Row {idx}: Bemerkungen = '{row_data['Bemerkungen']}'\n")
                            log_file.write(f"  Full row: {row_data.to_dict()}\n")
                    print(f"  → CONFLICT: {len(filled_indices)} rows with different Bemerkungen (logged, rows kept)")
                    # DO NOT add to rows_to_delete - keep all rows
        
        # Case 2 & 3: All rows have filled Bemerkungen OR all rows are empty
        else:
            if filled_indices:
                # All have filled Bemerkungen
                filled_values = [combined_df.loc[idx, 'Bemerkungen'] for idx in filled_indices]
                
                # Check if all values are identical
                if len(set(filled_values)) == 1:
                    # Case 2: All identical, delete all but the last one
                    rows_to_delete.extend(indices[:-1])
                    print(f"  → Case 2: All Bemerkungen identical, keeping last row only")
                else:
                    # Case 3: Different values - log conflict but DO NOT DELETE
                    conflicts_found = True
                    with open(duplicates_log_path, 'a', encoding='utf-8') as log_file:
                        log_file.write(f"\nConflict at Date: {date}, AM/PM: {am_pm}\n")
                        for idx in indices:
                            row_data = combined_df.loc[idx]
                            log_file.write(f"  Row {idx}: Bemerkungen = '{row_data['Bemerkungen']}'\n")
                            log_file.write(f"  Full row: {row_data.to_dict()}\n")
                    print(f"  → Case 3: CONFLICT - Different Bemerkungen values (logged, rows kept)")
                    # DO NOT add to rows_to_delete - keep all rows
            else:
                # All are empty, just keep the last one
                rows_to_delete.extend(indices[:-1])
                print(f"  → All Bemerkungen empty, keeping last row only")

# Remove duplicate rows
if rows_to_delete:
    combined_df = combined_df.drop(index=rows_to_delete)
    combined_df = combined_df.reset_index(drop=True)
    print(f"\n✓ Removed {len(rows_to_delete)} duplicate row(s)")
else:
    print(f"\n✓ No duplicates found")

# Report on conflicts
if conflicts_found:
    print(f"\n⚠ Some duplicates had conflicting Bemerkungen values.")
    print(f"Check '{duplicates_log_path}' for details.")
else:
    print(f"\n✓ No conflicts in duplicate Bemerkungen values")

print(f"\n{'='*60}")
print(f"FINAL DATA SUMMARY")
print(f"{'='*60}")
print(f"Total rows after deduplication: {len(combined_df)}")
print(f"\nFirst 10 rows of final data:")
print(combined_df.head(10))

✓ Sorted data by Date and AM/PM

Processing duplicates for Date: 2021-05-17 00:00:00, AM/PM: am (3 rows)
  → Case 2: All Bemerkungen identical, keeping last row only

Processing duplicates for Date: 2021-05-17 00:00:00, AM/PM: pm (3 rows)
  → Case 2: All Bemerkungen identical, keeping last row only

Processing duplicates for Date: 2021-05-18 00:00:00, AM/PM: am (3 rows)
  → Case 2: All Bemerkungen identical, keeping last row only

Processing duplicates for Date: 2021-05-18 00:00:00, AM/PM: pm (3 rows)
  → Case 2: All Bemerkungen identical, keeping last row only

Processing duplicates for Date: 2021-05-19 00:00:00, AM/PM: am (3 rows)
  → Case 2: All Bemerkungen identical, keeping last row only

Processing duplicates for Date: 2021-05-19 00:00:00, AM/PM: pm (3 rows)
  → Case 2: All Bemerkungen identical, keeping last row only

Processing duplicates for Date: 2021-05-20 00:00:00, AM/PM: am (3 rows)
  → Case 2: All Bemerkungen identical, keeping last row only

Processing duplicates for Dat

In [15]:
combined_df.to_csv('./private/combined_output.csv', index=False)